# Демо 2. Шаблони розмови і підготовка даних для SFT

В демо 1 донавчання моделі було повʼязане з заміною голови, а отже частковою зміною архітектури нейронів. В багатьох випадках при донавчанні LLM моделей архітектура не змінюється, а змінюється радше формат введених і виведених данних. Для цього потрібно зрозуміти, **у якому вигляді модель бачить діалог**.
В будь-якому випадку фізично на вхід моделі подається тензор з токенами, а за те як з цих токенів утворюються семантичні повідомлення відповідають **шаблони розмови** (chat templates).
Під час донавчання модель може вивчити новий шаблон розмови.

Донавчання LLM методом SFT виконується на діалогах, відформатованих шаблоном розмови.
У цьому ноутбуці розглянемо, що таке шаблон розмови і як підготувати дані у форматі діалогів;
саме донавчання виконаємо в демо 3.

План:
1. Завантажити base- та instruct-версії SmolLM3.
2. Подивитися, як шаблон розмови перетворює різні типи діалогів на текст.
3. Порівняти відповіді base- та instruct-моделі.
4. Перевірити режими міркування `/think` і `/no_think`.
5. Порівняти шаблони різних моделей.
6. Підготувати датасет для SFT у форматі `messages`.

Демо за мотивами вправ курсу [smol-course](https://huggingface.co/learn/smol-course) і розділу 11
[Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter11/1).
Для запуску в Colab оберіть середовище з GPU: *Runtime → Change runtime type → T4 GPU*.

In [1]:
# ⏱ Додано для питання 11: вимірювання часу виконання кожної клітинки.
# Після кожної клітинки друкується її тривалість; підсумок виводить функція timing_report() у розділі з відповідями.
import time
from IPython import get_ipython

NOTEBOOK_START = time.time()
CELL_TIMES = []  # (перший рядок клітинки, тривалість у секундах)
_timer = {"t0": None}


def _pre_run_cell(info):
    _timer["t0"] = time.time()


def _post_run_cell(result):
    if _timer["t0"] is None:
        return
    seconds = time.time() - _timer["t0"]
    _timer["t0"] = None
    lines = (getattr(result.info, "raw_cell", "") or "").strip().splitlines()
    CELL_TIMES.append((lines[0][:70] if lines else "", seconds))
    print(f"⏱ {seconds:.1f} с")


ip = get_ipython()
for _event, _callback in globals().get("_timing_callbacks", []):  # повторний запуск не дублює обробники
    try:
        ip.events.unregister(_event, _callback)
    except ValueError:
        pass
_timing_callbacks = [("pre_run_cell", _pre_run_cell), ("post_run_cell", _post_run_cell)]
for _event, _callback in _timing_callbacks:
    ip.events.register(_event, _callback)


def timing_report(top=5):
    """Друкує сумарний час усіх клітинок і повертає таблицю з найдовшими."""
    import pandas as pd

    df = pd.DataFrame(CELL_TIMES, columns=["Клітинка (перший рядок)", "Час, с"])
    total = df["Час, с"].sum()
    print(f"Клітинок виконано: {len(df)}")
    print(f"Сумарний час виконання клітинок: {total / 60:.1f} хв ({total:.0f} с)")
    print(f"Від запуску ноутбука минуло: {(time.time() - NOTEBOOK_START) / 60:.1f} хв (разом із паузами між клітинками)")
    return df.sort_values("Час, с", ascending=False).head(top).round(1)

In [2]:
# Встановлюємо потрібні бібліотеки; -q вимикає зайвий вивід під час встановлення
!pip install -q "transformers>=4.55.0" "datasets" "accelerate"

⏱ 9.3 с


In [3]:
import warnings

import torch
from datasets import get_dataset_split_names, load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

if torch.cuda.is_available():
    device = "cuda"
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Памʼять GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    device = "mps"
    print("Apple MPS")
else:
    device = "cpu"
    warnings.warn("Обчислення на CPU: генерація буде дуже повільною, потрібна GPU")

GPU: Tesla T4
Памʼять GPU: 15.6 GB
⏱ 18.0 с


In [4]:
# Детальна інформація про GPU NVIDIA: модель, зайнята й вільна памʼять
if device == "cuda":
    !nvidia-smi

Sun Nov  8 17:43:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Завантаження моделей

[**SmolLM3**](https://huggingface.co/blog/smollm3) — відкрита модель від Hugging Face на 3 млрд параметрів. Вона має дві версії:
- **Base** — лише після попереднього тренування: продовжує текст, але не веде діалог;
- **Instruct** (`SmolLM3-3B`) — донавчена виконувати інструкції й вести діалог.

In [5]:
base_model_name = "HuggingFaceTB/SmolLM3-3B-Base"
instruct_model_name = "HuggingFaceTB/SmolLM3-3B"

base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
instruct_tokenizer = AutoTokenizer.from_pretrained(instruct_model_name)

# Ваги SmolLM3 збережено в bfloat16, але T4 не підтримує його апаратно,
# тому завантажуємо у float16.
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name, dtype=torch.float16, device_map="auto"
)
instruct_model = AutoModelForCausalLM.from_pretrained(
    instruct_model_name, dtype=torch.float16, device_map="auto"
)
print("Моделі завантажено")

config.json:   0%|          | 0.00/943 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/151 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

... (12 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

Моделі завантажено
⏱ 210.4 с


## 2. Як шаблон розмови форматує діалог

Діалог задається списком повідомлень із ролями:
- `system` — інструкції для моделі (роль, стиль, обмеження);
- `user` — повідомлення користувача;
- `assistant` — відповіді моделі.

Метод `apply_chat_template` перетворює цей список на рядок зі спеціальними токенами.

Параметр `add_generation_prompt`:
- `False` — для передачі в модель завершених діалогів, тобто для **тренувальних даних**;
- `True` — додає в кінець заголовок відповіді асистента, щоб модель почала відповідати; використовується для **інференсу**.

Для SmolLM3 у системне повідомлення можна додати прапорець `/think` або `/no_think`,
який вмикає чи вимикає режим міркування.

In [6]:
# Різні типи діалогів для порівняння
conversations = {
    "simple_qa": [
        {"role": "system", "content": "/no_think"},
        {"role": "user", "content": "What is machine learning?"},
    ],
    "with_system": [
        {
            "role": "system",
            "content": "You are a helpful AI assistant specialized in explaining technical concepts clearly. /no_think",
        },
        {"role": "user", "content": "What is machine learning?"},
    ],
    "multi_turn": [
        {"role": "system", "content": "You are a math tutor. /no_think"},
        {"role": "user", "content": "What is calculus?"},
        {
            "role": "assistant",
            "content": "Calculus is a branch of mathematics that deals with rates of change and accumulation of quantities.",
        },
        {"role": "user", "content": "Can you give me a simple example?"},
    ],
    "reasoning_task": [
        {"role": "system", "content": "/think"},
        {
            "role": "user",
            "content": "Solve step by step: If a train travels 120 miles in 2 hours, what is its average speed?",
        },
    ],
}

for conv_type, messages in conversations.items():
    print(f"--- {conv_type.upper()} ---")

    # Завершений діалог: так виглядають тренувальні дані
    formatted_complete = instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    # Діалог для інференсу: у кінці додано заголовок відповіді асистента
    formatted_prompt = instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    print("Завершений діалог (add_generation_prompt=False):")
    print(formatted_complete)
    print("\nДля інференсу (add_generation_prompt=True):")
    print(formatted_prompt)
    print("\n" + "=" * 50 + "\n")

--- SIMPLE_QA ---
Завершений діалог (add_generation_prompt=False):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 08 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

<|im_start|>user
What is machine learning?<|im_end|>


Для інференсу (add_generation_prompt=True):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 08 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

<|im_start|>user
What is machine learning?<|im_end|>
<|im_start|>assistant
<think>

</think>



--- WITH_SYSTEM ---
Завершений діалог (add_generation_prompt=False):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 08 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant specialized in explaining technical concepts clearly.

<|im_s

**Що варто помітити:**
- Шаблон SmolLM3 сам додає до системного повідомлення блок метаданих (дата відсічення знань, поточна дата, режим міркування)
  і стандартну інструкцію, якщо власної немає. Модель бачить більше, ніж ми написали.
- Кожне повідомлення обгорнуте токенами `<|im_start|>роль` … `<|im_end|>` (формат ChatML).
- З `add_generation_prompt=True` рядок закінчується на `<|im_start|>assistant`: з цього місця модель починає генерувати.

## 3. Base проти Instruct

Ставимо обом моделям одне питання. Base-модель отримує сирий текст без шаблону,
instruct-модель — діалог, відформатований шаблоном розмови.

In [7]:
def generate(model, tokenizer, text, max_new_tokens=150, temperature=0.7):
    """Генерує продовження тексту й повертає лише нові токени, без промпту."""
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Відкидаємо токени промпту: декодуємо лише те, що згенерувала модель
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

⏱ 0.0 с


In [8]:
test_prompt = "Explain me, what is quantum computing?"

# Base-модель: сирий текст без шаблону
print("🤖 BASE-МОДЕЛЬ:")
print("-" * 50)
print(generate(base_model, base_tokenizer, test_prompt))

# Instruct-модель: діалог, відформатований шаблоном розмови
instruct_messages = [
    {"role": "system", "content": "/no_think"},
    {"role": "user", "content": test_prompt},
]
instruct_formatted = instruct_tokenizer.apply_chat_template(
    instruct_messages, tokenize=False, add_generation_prompt=True
)
print("\n" + "=" * 50)
print("💬 INSTRUCT-МОДЕЛЬ:")
print("-" * 50)
print(generate(instruct_model, instruct_tokenizer, instruct_formatted))

🤖 BASE-МОДЕЛЬ:
--------------------------------------------------


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


 What are the applications of quantum computing? Is it really helpful in the future? What is the difference between quantum computer and a classical computer? If you are curious about these questions, then read this article till the end. This article will help you to clear all your doubts about quantum computing and its applications.
Quantum computing is a kind of computer that works on the principle of quantum mechanics. It is a new type of computer that can solve problems that are beyond the capabilities of classical computers. Quantum computing is a field of research that is still in its early stages, but it has the potential to revolutionize the way we think about computers and how they work.
What is Quantum Computing?
Quantum computing is a new type of computer that works on the

💬 INSTRUCT-МОДЕЛЬ:
--------------------------------------------------
Quantum computing is a type of computing that uses the principles of quantum mechanics to perform calculations. It's a new and revolut

## 4. Режими міркування `/think` і `/no_think`

SmolLM3 — гібридна модель: у режимі `/think` вона спочатку генерує міркування в блоці `<think>…</think>`,
а потім відповідь; у режимі `/no_think` блок міркувань порожній.

In [9]:
reasoning_prompts = [
    "What is 15 × 24? Show your work.",
    "A recipe calls for 2 cups of flour for 12 cookies. How much flour is needed for 30 cookies?",
    "If I have $50 and spend $18.75 on lunch and $12.30 on a book, how much money do I have left?",
]

for thinking_flag in ["/no_think", "/think"]:
    print(f"===== Режим: {thinking_flag} =====\n")
    for i, prompt in enumerate(reasoning_prompts, 1):
        messages = [
            {"role": "system", "content": thinking_flag},
            {"role": "user", "content": prompt},
        ]
        formatted = instruct_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        # Нижча температура робить відповіді на арифметичні задачі стабільнішими
        answer = generate(instruct_model, instruct_tokenizer, formatted, max_new_tokens=300, temperature=0.3)
        print(f"Задача {i}: {prompt}")
        print(f"Відповідь: {answer}")
        print("\n" + "-" * 50 + "\n")

===== Режим: /no_think =====

Задача 1: What is 15 × 24? Show your work.
Відповідь: To solve the multiplication problem 15 × 24, we can break it down into smaller, more manageable steps.

First, let's multiply 15 by 4 (the units digit of 24):
15 × 4 = 60

Next, we multiply 15 by 20 (the tens digit of 24, which is equivalent to multiplying by 2 and then shifting the result one place to the left):
15 × 20 = 15 × 2 × 10 = 30 × 10 = 300

Now, we add the two results together:
60 + 300 = 360

So, 15 × 24 = 360.

Alternatively, we can also use the standard multiplication algorithm:

```
   15
× 24
-----
   60  (15 × 4)
+300  (15 × 20)
-----
  360
```

Both methods yield the same result: 15 × 24 = 360.

--------------------------------------------------

Задача 2: A recipe calls for 2 cups of flour for 12 cookies. How much flour is needed for 30 cookies?
Відповідь: To determine how much flour is needed for 30 cookies, we can start by finding out how much flour is required per cookie. 

The rec

## 5. У кожної моделі свій шаблон

Ролі `system`, `user`, `assistant` — це домовленість на рівні формату `messages`.
У токенах кожна модель записує їх по-своєму. Порівняємо шаблони кількох моделей.

In [10]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hi!"},
    {"role": "assistant", "content": "Hello! How can I help?"},
]

for name in ["HuggingFaceTB/SmolLM3-3B", "Qwen/Qwen3-0.6B", "microsoft/Phi-3.5-mini-instruct"]:
    tok = AutoTokenizer.from_pretrained(name)
    print(f"===== {name} =====")
    print(tok.apply_chat_template(messages, tokenize=False))
    print()

===== HuggingFaceTB/SmolLM3-3B =====
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 08 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a helpful assistant.

<|im_start|>user
Hi!<|im_end|>
<|im_start|>assistant
Hello! How can I help?<|im_end|>




config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

... (7 outputs left out)

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

===== microsoft/Phi-3.5-mini-instruct =====
<|system|>
You are a helpful assistant.<|end|>
<|user|>
Hi!<|end|>
<|assistant|>
Hello! How can I help?<|end|>
<|endoftext|>

⏱ 6.0 с


Прапорці режиму міркування також залежать від моделі. У Qwen3, наприклад,
режим вмикається параметром `enable_thinking` у `apply_chat_template`.

In [11]:
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
question = [{"role": "user", "content": "What is 2 + 2?"}]

for enable_thinking in [True, False]:
    print(f"===== enable_thinking={enable_thinking} =====")
    print(qwen_tokenizer.apply_chat_template(
        question, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking
    ))

===== enable_thinking=True =====
<|im_start|>user
What is 2 + 2?<|im_end|>
<|im_start|>assistant

===== enable_thinking=False =====
<|im_start|>user
What is 2 + 2?<|im_end|>
<|im_start|>assistant
<think>

</think>


⏱ 0.7 с


### А що з base-моделлю?

Base-модель не донавчали на діалогах, тому шаблон у її токенізаторі або відсутній, або модель ніколи не бачила його токенів у ролі розмітки діалогу.
Перед SFT шаблон для base-моделі задають самостійно — найпростіше взяти його з instruct-версії тієї самої моделі.

In [12]:
print("Шаблон є в base-токенізаторі:  ", base_tokenizer.chat_template is not None)
print("Шаблон є в instruct-токенізаторі:", instruct_tokenizer.chat_template is not None)

Шаблон є в base-токенізаторі:   False
Шаблон є в instruct-токенізаторі: True
⏱ 0.0 с


## 6. Підготовка датасету для SFT

Для SFT дані зберігають у форматі `messages` — тобто у списку повідомлень із ролями.
Бібліотека TRL (`SFTTrainer`) сама застосовує до нього шаблон розмови під час тренування.

### SmolTalk2

**SmolTalk2** — датасет, на якому донавчали SmolLM3. Він складається з багатьох частин (splits) з різних джерел.
Список частин можна отримати без завантаження самих даних.

In [13]:
splits = get_dataset_split_names("HuggingFaceTB/smoltalk2", "SFT")
print(f"Кількість частин: {len(splits)}")
for s in splits:
    print(" -", s)

README.md:   0%|          | 0.00/23.5k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/124 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Кількість частин: 25
 - LongAlign_64k_Qwen3_32B_yarn_131k_think
 - OpenThoughts3_1.2M_think
 - aya_dataset_Qwen3_32B_think
 - multi_turn_reasoning_if_think
 - s1k_1.1_think
 - smolagents_toolcalling_traces_think
 - smoltalk_multilingual8_Qwen3_32B_think
 - smoltalk_systemchats_Qwen3_32B_think
 - table_gpt_Qwen3_32B_think
 - LongAlign_64k_context_lang_annotated_lang_6_no_think
 - Mixture_of_Thoughts_science_no_think
 - OpenHermes_2.5_no_think
 - OpenThoughts3_1.2M_no_think_no_think
 - hermes_function_calling_v1_no_think
 - smoltalk_multilingual_8languages_lang_5_no_think
 - smoltalk_smollm3_everyday_conversations_no_think
 - smoltalk_smollm3_explore_instruct_rewriting_no_think
 - smoltalk_smollm3_smol_magpie_ultra_no_think
 - smoltalk_smollm3_smol_rewrite_no_think
 - smoltalk_smollm3_smol_summarize_no_think
 - smoltalk_smollm3_systemchats_30k_no_think
 - table_gpt_no_think
 - tulu_3_sft_personas_instruction_following_no_think
 - xlam_traces_no_think
 - smoltalk_everyday_convs_reasoning_

In [14]:
# Завантажуємо лише файл однієї частини через data_files: повний датасет займає десятки гігабайт,
# а load_dataset(..., split=...) спершу перебирає файли всіх частин і працює дуже довго
split = "smoltalk_everyday_convs_reasoning_Qwen3_32B_think"
everyday = load_dataset("HuggingFaceTB/smoltalk2", data_files=f"SFT/{split}-*.parquet", split="train")
print(everyday)
print()
for message in everyday[0]["messages"]:
    print(f"[{message['role']}] {message['content'][:200]}")

SFT/smoltalk_everyday_convs_reasoning_Qw(…): reconstructing file:   0%|          |  0.00B / 6.33MB            

SFT/smoltalk_everyday_convs_reasoning_Qw(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages', 'chat_template_kwargs', 'source'],
    num_rows: 2057
})

[user] Hi there
[assistant] <think>
Okay, the user sent "Hi there". That's a friendly greeting. I should respond in a welcoming way. Let me check the guidelines. I need to be helpful, keep the conversation going, and maybe ask h
[user] I'm looking for a healthy breakfast idea. What's a good option?
[assistant] <think>
Okay, the user is asking for a healthy breakfast idea. Let me think about what makes a breakfast healthy. It should be balanced, providing a mix of nutrients like protein, fiber, healthy fats,
⏱ 1.4 с


Ця частина зібрана для режиму `/think`, тому відповіді асистента містять блок міркувань `<think>…</think>`.

Як видно з назви частини, відповіді в ній згенерувала модель Qwen3-32B. Такі **синтетичні дані** від сильнішої моделі —
поширений спосіб отримати тисячі прикладів для SFT без ручного написання відповідей. Помилки моделі-генератора при цьому переходять у дані,
тому синтетичні дані фільтрують.

### Перетворення власного датасету у формат `messages`

Більшість датасетів має інший формат, наприклад колонки «питання» і «відповідь».
Перетворимо датасет задач з математики **GSM8K** у формат `messages`.

In [15]:
gsm8k = load_dataset("openai/gsm8k", "main", split="train[:100]")  # невелика вибірка для демо
print("Оригінальний приклад:", gsm8k[0])


def to_messages(examples):
    """Перетворює пари «питання — відповідь» на діалоги з ролями."""
    processed = []
    for question, answer in zip(examples["question"], examples["answer"]):
        processed.append([
            {"role": "system", "content": "You are a math tutor. Solve problems step by step."},
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ])
    return {"messages": processed}


gsm8k_messages = gsm8k.map(to_messages, batched=True, remove_columns=gsm8k.column_names)
print("\nУ форматі messages:", gsm8k_messages[0])

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Оригінальний приклад: {'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}


Map:   0%|          | 0/100 [00:00<?, ? examples/s]


У форматі messages: {'messages': [{'content': 'You are a math tutor. Solve problems step by step.', 'role': 'system'}, {'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'role': 'user'}, {'content': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72', 'role': 'assistant'}]}
⏱ 2.3 с


### Що побачить модель під час тренування

Застосуємо шаблон розмови до підготовленого прикладу. Саме цей рядок модель отримає під час SFT.

In [16]:
text = instruct_tokenizer.apply_chat_template(
    gsm8k_messages[0]["messages"],
    tokenize=False,
    add_generation_prompt=False,  # діалог завершений: відповідь асистента вже є
)
print(text)
print(f"Довжина в токенах: {len(instruct_tokenizer(text)['input_ids'])}")

<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 08 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a math tutor. Solve problems step by step.

<|im_start|>user
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>
<|im_start|>assistant
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72<|im_end|>

Довжина в токенах: 145
⏱ 0.0 с


**Що варто помітити:** шаблон знову додав метадані й режим `/think`, хоча в даних їх немає,
а відповідь асистента не містить блоку міркувань. Якщо донавчати на таких прикладах, модель вчитиметься
відповідати без міркувань навіть у режимі `/think`. Тому формат даних і шаблон мають узгоджуватися з тим,
як модель використовуватиметься після донавчання.

---
# Відповіді на теоретичні питання (Лабораторна 5, п. 0.1, питання 11–21)

Клітинки з кодом у цьому розділі самі збирають дані з вашого запуску. Усі числа в текстових відповідях узято з виводів цього запуску (Google Colab, GPU Tesla T4).

## Питання 11. Як довго виконувалися клітинки? Чи були проблеми з запуском?

In [17]:
# Тривалість виконання клітинок ноутбука (дані збирає обробник із першої клітинки)
import torch

print("Пристрій:", torch.cuda.get_device_name() if torch.cuda.is_available() else "GPU не використовувалась (CPU)")
timing_report()

Пристрій: Tesla T4
Клітинок виконано: 15
Сумарний час виконання клітинок: 5.4 хв (321 с)
Від запуску ноутбука минуло: 5.4 хв (разом із паузами між клітинками)


,Клітинка (перший рядок),"Час, с"
3,"base_model_name = ""HuggingFaceTB/SmolLM3-3B-Base""",210.4
7,reasoning_prompts = [,57.7
1,import warnings,18.0
6,"test_prompt = ""Explain me, what is quantum com...",13.9
0,# Встановлюємо потрібні бібліотеки; -q вимикає...,9.3


⏱ 0.0 с


**Відповідь.** Усі клітинки виконалися за **5,4 хв (321 с)** на **Tesla T4**. Найдовше (**210,4 с**, тобто ≈65% усього часу) тривало завантаження двох моделей SmolLM3-3B (Base й Instruct, по ≈6 GB у fp16); далі клітинка з генерацією в режимах `/no_think` і `/think` (57,7 с), клітинка з імпортами (18,0 с), порівняння Base й Instruct (13,9 с) та встановлення бібліотек (9,3 с).

**Проблеми з запуском:** помилок немає, `CUDA out of memory` теж не було: дві моделі по ≈6 GB вмістилися в T4 (15,6 GB). З'являлися лише попередження:
- про неавтентифіковані запити до Hugging Face Hub (`HF_TOKEN`), не критично;
- про `clean_up_tokenization_spaces` для BPE-токенізатора: інформаційне повідомлення, на результат не впливає;
- про поле `rope_parameters` у конфігурації Qwen3-0.6B: рекомендація щодо формату конфігурації, не помилка.

## Питання 12. Для чого змінювати `add_generation_prompt` і як за готовою розмовою дізнатися його значення?

**Відповідь.** Параметр визначає, чи додасть шаблон у кінець рядка **заголовок наступної відповіді асистента**:
- `add_generation_prompt=True` — **інференс**. Діалог закінчується відкритим заголовком `<|im_start|>assistant` (у режимі `/no_think` разом із порожнім блоком `<think></think>`), тож модель розуміє, що тепер її черга, і починає генерувати відповідь. Без цього заголовка модель могла б продовжити репліку користувача або написати щось довільне.
- `add_generation_prompt=False` — **тренувальні дані**. Діалог уже завершений: відповідь асистента є в даних, і шаблон просто закриває її токеном `<|im_end|>`. Додатковий порожній заголовок асистента тут був би зайвим і навчив би модель хибному.

**Як визначити значення за вже згенерованою розмовою:** подивитися на **кінець рядка**. У моєму виводі:
- при `add_generation_prompt=False` рядок закінчується закритим повідомленням користувача: `…What is machine learning?<|im_end|>\n`;
- при `add_generation_prompt=True` до нього додається рівно `<|im_start|>assistant\n<think>\n\n</think>\n`: відкритий заголовок асистента без тексту й без `<|im_end|>`, а також порожній блок міркувань, бо режим `/no_think`.

Тож якщо діалог закінчується заголовком асистента (з порожнім `<think>` або без нього), то було `True`, а якщо закритим `<|im_end|>` останнього повідомлення, то `False`. Найнадійніше відтворити розмову з обома значеннями й порівняти з рядком, що й робить функція `guess_add_generation_prompt` нижче (на моєму запуску вона повернула `True` і `False` відповідно до очікування).

In [18]:
# Питання 12: чим відрізняються рядки з add_generation_prompt=False і True
qa_messages = conversations["simple_qa"]
off = instruct_tokenizer.apply_chat_template(qa_messages, tokenize=False, add_generation_prompt=False)
on = instruct_tokenizer.apply_chat_template(qa_messages, tokenize=False, add_generation_prompt=True)

print("Кінець рядка при add_generation_prompt=False:", repr(off[-60:]))
print("Кінець рядка при add_generation_prompt=True: ", repr(on[-60:]))
if on.startswith(off):
    print("\nТе, що додає add_generation_prompt=True:", repr(on[len(off):]))


def guess_add_generation_prompt(rendered, messages, tokenizer):
    """Вгадує значення add_generation_prompt за готовим рядком: звіряє його з обома варіантами шаблону."""
    off = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    on = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    if rendered == on:
        return True
    if rendered == off:
        return False
    return None  # рядок не збігається з жодним варіантом


print("\nВгадуємо для рядка on  (очікуємо True): ", guess_add_generation_prompt(on, qa_messages, instruct_tokenizer))
print("Вгадуємо для рядка off (очікуємо False):", guess_add_generation_prompt(off, qa_messages, instruct_tokenizer))

Кінець рядка при add_generation_prompt=False: 'Face.\n\n<|im_start|>user\nWhat is machine learning?<|im_end|>\n'
Кінець рядка при add_generation_prompt=True:  'learning?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n'

Те, що додає add_generation_prompt=True: '<|im_start|>assistant\n<think>\n\n</think>\n'

Вгадуємо для рядка on  (очікуємо True):  True
Вгадуємо для рядка off (очікуємо False): False
⏱ 0.0 с


## Питання 13. Чим відрізняються ролі `system` і `assistant`?

**Відповідь.**
- **`system`** — інструкція **розробника** для моделі: роль, стиль, формат, обмеження («You are a math tutor», «відповідай лише JSON»). Це не репліка співрозмовника: вона стоїть на початку діалогу, діє на всю розмову, зазвичай прихована від користувача, і модель вчать надавати їй вищий пріоритет, ніж повідомленням користувача. У SmolLM3 у системне повідомлення також потрапляють метадані й прапорець `/think` / `/no_think`.
- **`assistant`** — **відповіді самої моделі**. У даних для інференсу це попередні відповіді моделі (історія діалогу чи few-shot приклади), а в даних для SFT це **цільовий текст, який модель вчиться генерувати**: саме на токенах цієї ролі рахується loss (див. питання 29).

Коротко: `system` задає правила гри, `user` ставить питання, `assistant` відповідає. Модель під час SFT вчиться передбачати лише токени третьої ролі.

## Питання 14. Прокоментуйте вивід секції 2 «Як шаблон розмови форматує діалог»

**Відповідь.** Шаблон SmolLM3 перетворює список `messages` на **один рядок**. З виводу видно, що шаблонізатор додав те, чого не було в наших повідомленнях:

1. **Системний блок із метаданими:** `Knowledge Cutoff Date: June 2025`, `Today Date: 08 November 2026` (дата підставляється автоматично, це день запуску) і `Reasoning Mode: /no_think` (береться з нашого прапорця). Модель сама не знає сьогоднішньої дати, а режим міркування вмикається саме цим рядком.
2. **Розділ `Custom Instructions`.** Для `simple_qa`, де власного системного тексту не було, підставлена типова інструкція `You are a helpful AI assistant named SmolLM, trained by Hugging Face.`, а для `with_system` і `multi_turn` у цьому розділі стоять наші власні тексти («…explaining technical concepts clearly», «You are a math tutor.»).
3. **Службові токени** `<|im_start|>роль … <|im_end|>` навколо повідомлень. Системний блок у цьому шаблоні не закривається `<|im_end|>`, а просто закінчується порожнім рядком перед `<|im_start|>user`. Токен `<|im_end|>` слугує й сигналом, коли припинити генерацію.
4. **Історія діалогу** в `multi_turn`: попередня відповідь асистента лишається в рядку, і навіть у ній шаблон додав порожній блок `<think>\n\n</think>` (режим `/no_think`).
5. **Хвіст для генерації** (лише у версії з `add_generation_prompt=True`): `<|im_start|>assistant\n<think>\n\n</think>\n`, тобто заголовок асистента і порожній блок міркувань. Це підказує моделі відповідати одразу, без міркувань (питання 12).

Висновок: те, що для нас є «розмовою», для моделі є послідовністю токенів зі спеціальною розміткою, тому формат треба відтворювати точно так, як під час навчання моделі.

## Питання 15. Різниця між Base і Instruct у секції 3

**Відповідь.** Обидві моделі мають однакову архітектуру й навіть спільне попереднє тренування, але різну **поведінку**:
- **Base** лише продовжує текст за статистикою інтернету. Питання «Explain me, what is quantum computing?» вона сприймає як початок документа, тому часто **продовжує перелік схожих питань**, повторюється, відповідає уривчасто або обривається на середині речення (генерація найчастіше зупиняється лише на ліміті `max_new_tokens`, бо модель не вчилася завершувати відповідь токеном `<|im_end|>`).
- **Instruct** пройшла SFT (і подальше вирівнювання) на діалогах, тому розуміє роль assistant: дає структуровану відповідь і вміє завершити її токеном `<|im_end|>` (у моєму виводі її відповідь теж обрізана лімітом 150 токенів, бо була довгою).

Ще одна різниця в підготовці входу: Base отримує **сирий текст**, Instruct отримує **діалог, відформатований шаблоном розмови**. Саме SFT перетворює першу модель на другу, і це те, що ми робитимемо в демо 3 на власній задачі.

**Що було у моєму виводі.** На запит «Explain me, what is quantum computing?» **base-модель не відповіла, а продовжила текст:** спочатку дописала ще кілька питань («What are the applications of quantum computing? Is it really helpful…? What is the difference between…?»), потім перейшла в стиль вступу до статті («If you are curious about these questions, then read this article till the end…») і повторювала одне й те саме означення, обірвавшись на півслові на ліміті 150 токенів. **Instruct-модель** одразу дала означення квантових обчислень, а далі почала нумерований список ключових понять (qubits, entanglement, quantum gates).

## Питання 16. Чому в режимі `/think` передано `max_new_tokens=300`?

**Відповідь.** У режимі `/think` модель **спочатку генерує міркування** у блоці `<think>…</think>` і лише потім дає відповідь. Міркування займають сотні токенів, і всі вони лічаться в ліміті `max_new_tokens`: за стандартних 150 токенів генерація обірвалась би **всередині міркування**, і відповіді не було б узагалі. У `/no_think` блок міркувань порожній, тож відповідь коротша. Тому для `/think` ліміт підвищено до 300.

**Проте мій запуск показав, що й 300 токенів замало.** У таблиці з питання 17 усі три відповіді `/think` досягли ліміту (300 із 300 токенів) і обірвались посеред міркування, не дійшовши до остаточної відповіді (у виводі секції 4 вони закінчуються на півслові: «Let me add them:», «Starting from the right: 0 - 5.»). Ті самі задачі в режимі `/no_think` вклались у 98–170 токенів. Висновок: ліміт слід задавати з урахуванням режиму й складності задачі. Для `/think` на таких задачах потрібно значно більше (наприклад, 1000 і більше токенів; цього значення я не перевіряв), а сам режим помітно дорожчий і повільніший.

## Питання 17. Чи впоралася модель з математичними задачами? Який режим обрати?

Правильні відповіді (порахуйте самі): 15 × 24 = **360**; 2 склянки на 12 печив → на 30 печив 2/12 × 30 = **5 склянок**; 50 − 18,75 − 12,30 = **18,95 $**. Клітинка нижче повторює генерацію й автоматично перевіряє, чи є в **остаточній відповіді** правильне число, а також чи модель встигла завершити міркування в межах 300 токенів.

In [19]:
# Питання 17: автоматична перевірка арифметичних відповідей у двох режимах
import re

import pandas as pd

checks = [  # (задача, правильна відповідь для людини, регулярний вираз для пошуку числа)
    (reasoning_prompts[0], "360", r"(?<![\d.,])360(?!\d)"),
    (reasoning_prompts[1], "5 cups", r"(?<![\d.,])5(?:\.0+)?(?!\d)"),
    (reasoning_prompts[2], "$18.95", r"18\.95"),
]
LIMIT = 300
rows = []
for flag in ["/no_think", "/think"]:
    for prompt, human_answer, pattern in checks:
        formatted = instruct_tokenizer.apply_chat_template(
            [{"role": "system", "content": flag}, {"role": "user", "content": prompt}],
            tokenize=False, add_generation_prompt=True,
        )
        inputs = instruct_tokenizer(formatted, return_tensors="pt").to(instruct_model.device)
        with torch.no_grad():
            out = instruct_model.generate(
                **inputs, max_new_tokens=LIMIT, temperature=0.3, do_sample=True,
                pad_token_id=instruct_tokenizer.eos_token_id,
            )
        new_tokens = out[0][inputs["input_ids"].shape[1]:]
        raw = instruct_tokenizer.decode(new_tokens, skip_special_tokens=False)  # зберігаємо теги, щоб побачити </think>
        finished = flag == "/no_think" or "</think>" in raw
        final = raw.split("</think>")[-1] if "</think>" in raw else raw
        rows.append({
            "режим": flag,
            "задача": prompt[:40] + "…",
            "правильно": human_answer,
            "токенів згенеровано": len(new_tokens),
            "досягнуто ліміту": len(new_tokens) >= LIMIT,
            "міркування завершено": finished,
            "є правильне число": finished and bool(re.search(pattern, final)),
        })
math_table = pd.DataFrame(rows)
display(math_table)
print("Частка правильних відповідей:", math_table.groupby("режим")["є правильне число"].mean().round(2).to_dict())

,режим,задача,правильно,токенів згенеровано,досягнуто ліміту,міркування завершено,є правильне число
0,/no_think,What is 15 × 24? Show your work.…,360,170,False,True,True
1,/no_think,A recipe calls for 2 cups of flour for 1…,5 cups,153,False,True,True
2,/no_think,If I have $50 and spend $18.75 on lunch …,$18.95,98,False,True,True
3,/think,What is 15 × 24? Show your work.…,360,300,True,False,False
4,/think,A recipe calls for 2 cups of flour for 1…,5 cups,300,True,False,False
5,/think,If I have $50 and spend $18.75 on lunch …,$18.95,300,True,False,False


Частка правильних відповідей: {'/no_think': 1.0, '/think': 0.0}
⏱ 57.3 с


**Відповідь.** За таблицею вище (запуск із `temperature=0.3`):
- **`/no_think`: 3 з 3 правильно** (360; 5 cups; $18.95). Відповіді вклались у 98–170 токенів, із розписаними кроками;
- **`/think`: 0 з 3** з точки зору автоматичної перевірки, бо **усі три генерації впёрлися в ліміт 300 токенів** (колонка «досягнуто ліміту» — `True`, «міркування завершено» — `False`) і обірвались до остаточної відповіді. Прочитавши вивід секції 4, бачимо, що самі міркування були правильними (15 × 4 = 60 і 15 × 20 = 300; 2/12 = 1/6 склянки на печиво, 30 × 1/6 = 5; 18,75 + 12,30 = 31,05), але модель у режимі `/think` багато разів перевіряє себе й просто не встигає закінчити.

Отже, модель **з математичними задачами впоралась у `/no_think`**; у `/think` вона до відповіді не дійшла лише через ліміт токенів, а не через помилки в розв'язанні.

**Який режим обрав би я.** Для таких простих задач `/no_think`: він дав правильні відповіді з розписаними кроками й коштує щонайменше у 1,8–3 рази менше токенів (98–170 проти ≥300), тож швидший і дешевший. `/think` має сенс для складніших багатокрокових задач, але лише з набагато більшим `max_new_tokens` (це я не перевіряв). Готову відповідь у будь-якому разі варто перевіряти калькулятором: LLM помиляються в арифметиці, особливо з десятковими дробами. Для надійності краще підключати інструмент (калькулятор, виконання коду). Для чату, переписування тексту й SFT-класифікації `/no_think` краще завжди.

## Питання 18. Які відмінності в шаблонах розмов ви помітили в секції 5?

**Відповідь.** Ті самі повідомлення `system` / `user` / `assistant` у різних моделей перетворюються на різний текст:
- **SmolLM3** і **Qwen3** використовують схожий формат ChatML (`<|im_start|>роль … <|im_end|>`). Але SmolLM3 сама додає до системного повідомлення метадані (дати, режим міркування) й типову інструкцію і не закриває системний блок `<|im_end|>`. Qwen3 метаданих не додає: системне повідомлення лишається таким, як ми його передали, і закривається `<|im_end|>`, зате перед відповіддю асистента вона вставляє порожній блок `<think>\n\n</think>`.
- **Phi-3.5-mini** має зовсім інші маркери: `<|system|>`, `<|user|>`, `<|assistant|>`, а кінець повідомлення позначає токеном `<|end|>`.
- **Режим міркування** кодується по-різному: у SmolLM3 прапорцем `/think` / `/no_think` у системному повідомленні, у Qwen3 параметром `enable_thinking` у `apply_chat_template`. З `enable_thinking=False` шаблон додає до заголовка асистента порожній блок `<think></think>`, і модель одразу відповідає.

Отже, роль і зміст повідомлень однакові, а **токени розмітки й службові блоки залежать від моделі**. Шаблон від однієї моделі не можна просто перенести на іншу.

## Питання 19. Для чого знати про шаблони розмов, якщо ми хочемо донавчити модель методом SFT?

**Відповідь.** SFT вчить модель передбачати токени тексту, який **утворює шаблон**, тому шаблон є частиною тренувальних даних:
1. **Узгодженість навчання й інференсу.** Якщо донавчити в одному форматі, а на інференсі подати інший (чи без `add_generation_prompt`, чи з іншим режимом міркування), якість погіршується: модель бачить розподіл токенів, якого не вчилася.
2. **Стоп-токен.** Модель вчиться завершувати відповідь саме токеном із шаблону (`<|im_end|>`); без нього вона не зупиняється.
3. **Маскування loss.** Щоб рахувати loss лише на відповідях асистента (`assistant_only_loss`), треба знати, де в рядку починається й закінчується роль `assistant`.
4. **Режим міркування й службові блоки.** Як показує секція 6, шаблон додає метадані й режим `/think`, тож якщо дані не збігаються з цим режимом, модель вчитиметься поведінці, яку ми не планували.
5. **Base-модель** не має шаблону (або не бачила його токенів у ролі розмітки). Шаблон потрібно задати самим, найпростіше взяти його з instruct-версії тієї ж моделі. Інакше нові спеціальні токени матимуть нетреновані вбудовування.

## Питання 20. З яких даних побудовано датасет `HuggingFaceTB/smoltalk2`?

**Відповідь.** SmolTalk2 — датасет, на якому донавчали SmolLM3. Він **не є одним набором**, а складається з багатьох частин (splits) з різних джерел. Для конфігурації `SFT` їх **25**, з яких **10** призначено для режиму з міркуваннями (`think`) і **15** для режиму без них (`no_think`). За назвами частин у списку видно такі групи:
- **готові відкриті датасети:** `OpenThoughts3_1.2M`, `s1k_1.1`, `Mixture_of_Thoughts_science` (міркування, математика, наука), `OpenHermes_2.5`, `tulu_3_sft_personas_instruction_following` (виконання інструкцій), `hermes_function_calling_v1`, `xlam_traces` (виклик функцій), `table_gpt` (таблиці);
- **частини власного датасету SmolTalk** (`smoltalk_smollm3_…`: повсякденні розмови, `smol_magpie_ultra`, переписування, резюмування, системні промпти) та `smolagents_toolcalling_traces`;
- **синтетичні дані:** **6 частин** мають у назві `Qwen3_32B` (`LongAlign_64k…`, `aya_dataset…`, `smoltalk_multilingual8…`, `smoltalk_systemchats…`, `table_gpt…`, `smoltalk_everyday_convs_reasoning…`). Відповіді в них, разом із ланцюжком міркувань `<think>`, згенерувала сильніша модель Qwen3-32B;
- окремі частини для **довгого контексту** (`LongAlign_64k`) і **багатомовності** (`aya_dataset`, `smoltalk_multilingual…`).

Отже, це суміш відкритих датасетів і синтетичних відповідей сильнішої моделі, що покриває міркування, код і математику, виконання інструкцій, виклик функцій, таблиці, довгий контекст і кілька мов. Клітинка нижче підраховує це за реальним списком частин.

In [20]:
# Питання 20: що видно про склад SmolTalk2 за назвами частин (змінна splits створена в розділі 6)
think = [s for s in splits if "no_think" not in s and "think" in s]
no_think = [s for s in splits if "no_think" in s]
generated = [s for s in splits if "Qwen3_32B" in s]
print(f"Усього частин: {len(splits)}; з міркуваннями (think): {len(think)}; без міркувань (no_think): {len(no_think)}")
print(f"Частин, відповіді в яких згенерувала Qwen3-32B: {len(generated)}")
print("\nПерші 10 частин без Qwen3_32B у назві (ймовірно, готові відкриті датасети):")
for s in [s for s in splits if "Qwen3_32B" not in s][:10]:
    print(" -", s)

Усього частин: 25; з міркуваннями (think): 10; без міркувань (no_think): 15
Частин, відповіді в яких згенерувала Qwen3-32B: 6

Перші 10 частин без Qwen3_32B у назві (ймовірно, готові відкриті датасети):
 - OpenThoughts3_1.2M_think
 - multi_turn_reasoning_if_think
 - s1k_1.1_think
 - smolagents_toolcalling_traces_think
 - LongAlign_64k_context_lang_annotated_lang_6_no_think
 - Mixture_of_Thoughts_science_no_think
 - OpenHermes_2.5_no_think
 - OpenThoughts3_1.2M_no_think_no_think
 - hermes_function_calling_v1_no_think
 - smoltalk_multilingual_8languages_lang_5_no_think
⏱ 0.0 с


## Питання 21. Які модифікації даних відбуваються в секції «Перетворення власного датасету у формат messages»?

**Відповідь.** GSM8K має колонки `question` і `answer`. Функція `to_messages` (`dataset.map(..., batched=True)`) перетворює кожен рядок на **діалог із трьох повідомлень**:
1. `system`: додається нова системна інструкція «You are a math tutor. Solve problems step by step.» (її в оригінальних даних не було);
2. `user`: сюди потрапляє `question`;
3. `assistant`: сюди потрапляє `answer`.

Старі колонки видаляються (`remove_columns`), і лишається **єдина колонка `messages`** зі списком повідомлень, а це формат, який очікує `SFTTrainer`. Також береться невелика вибірка (`train[:100]`) для демонстрації. **Текст самих відповідей не змінюється**: у `answer` лишаються «калькуляторні» вставки `<<…>>` і фінальний маркер `#### число`. Клітинка нижче показує структуру до й після.

Вивід клітинки нижче це підтверджує: **100 прикладів**, колонки `['question', 'answer']` змінились на `['messages']`, ролі в першому прикладі `['system', 'user', 'assistant']`, а відповідь асистента збігається з оригінальним полем `answer` (`True`).

In [21]:
# Питання 21: структура даних до й після перетворення
print("GSM8K до:   ", gsm8k.column_names, "—", len(gsm8k), "прикладів")
print("GSM8K після:", gsm8k_messages.column_names)
print("\nРолі в першому прикладі:", [m["role"] for m in gsm8k_messages[0]["messages"]])
print("Відповідь асистента збігається з оригінальним полем answer:", gsm8k_messages[0]["messages"][2]["content"] == gsm8k[0]["answer"])

GSM8K до:    ['question', 'answer'] — 100 прикладів
GSM8K після: ['messages']

Ролі в першому прикладі: ['system', 'user', 'assistant']
Відповідь асистента збігається з оригінальним полем answer: True
⏱ 0.0 с
